# Assignment 4: Retrieval-augmented generation




## Preliminaries

Install the packages required by the assignment and the utilities used for evaluation.


In [1]:
%pip -q install -U langchain langchain-community langchain-huggingface langchain-core langchain-text-splitters langchain-chroma sentence-transformers transformers accelerate chromadb scikit-learn pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2

## Task 1.1 - Downloading and inspecting the question-answering dataset

- Download `ori_pqal.json` from PubMedQA.
- Keep only `yes` and `no` examples.
- Create `documents` from `CONTEXTS + LONG_ANSWER` and `questions` with the required gold fields.
- Print one question and one document as the sanity check.


In [2]:
from pathlib import Path
import json
import random
import re
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_URL = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json"
DATA_PATH = Path("/content/ori_pqal.json")

!wget -q -O "$DATA_PATH" "$DATA_URL"

tmp_data = pd.read_json(DATA_PATH).T
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])].copy()

documents = pd.DataFrame(
    {
        "abstract": tmp_data.apply(
            lambda row: " ".join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1
        ),
        "year": tmp_data.YEAR,
    }
)

questions = pd.DataFrame(
    {
        "question": tmp_data.QUESTION,
        "year": tmp_data.YEAR,
        "gold_label": tmp_data.final_decision,
        "gold_context": tmp_data.LONG_ANSWER,
        "gold_document_id": documents.index,
    }
)

assert len(questions) == len(documents)
assert set(questions.gold_label.unique()) <= {"yes", "no"}
assert questions.gold_document_id.astype(str).tolist() == documents.index.astype(str).tolist()

print(f"Retained yes/no questions: {len(questions):,}")
display(questions.head(3))
display(documents.head(3))

print("\nSanity-check question:\n", questions.iloc[0].question)
print("\nSanity-check document:\n", documents.iloc[0].abstract)


Retained yes/no questions: 890


,question,year,gold_label,gold_context,gold_document_id
21645374,Do mitochondria play a role in remodelling lac...,2011,yes,Results depicted mitochondrial dynamics in viv...,21645374
16418930,Landolt C and snellen e acuity: differences in...,2006,no,"Using the charts described, there was only a s...",16418930
9488747,"Syncope during bathing in infants, a pediatric...",1997,yes,"""Aquagenic maladies"" could be a pediatric form...",9488747


,abstract,year
21645374,Programmed cell death (PCD) is the regulated d...,2011
16418930,Assessment of visual acuity depends on the opt...,2006
9488747,Apparent life-threatening events in infants ar...,1997



Sanity-check question:
 Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Sanity-check document:
 Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages

## Task 2.1 - Select a language model

Load a Hugging Face generator with `HuggingFacePipeline.from_model_id`, set `return_full_text=False`, and invoke it once as a sanity check.


In [3]:
from langchain_huggingface.llms import HuggingFacePipeline

assert torch.cuda.is_available(), "In Colab, select Runtime > Change runtime type > GPU."

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

model = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    device=0,
    batch_size=8,
    model_kwargs={"torch_dtype": torch.float16},
    pipeline_kwargs={
        "max_new_tokens": 8,
        "do_sample": False,
        "return_full_text": False,
    },
)

sanity_prompt = "Answer with only Yes or No. Is water composed of hydrogen and oxygen?"
sanity_answer = model.invoke(sanity_prompt).strip()
print("Prompt:", sanity_prompt)
print("Model output:", sanity_answer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before p

Prompt: Answer with only Yes or No. Is water composed of hydrogen and oxygen?
Model output: Yes.

Yes, water is primarily composed


## Task 3.1 - Embedding model

Create `HuggingFaceEmbeddings`, embed one passage with `embed_query`, and confirm that the result has shape `(embedding_dim,)`.


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

example_embedding = np.asarray(
    embedding_model.embed_query("Programmed cell death is called apoptosis."),
    dtype=np.float32,
)

print("Embedding shape:", example_embedding.shape)
assert example_embedding.ndim == 1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (384,)


## Task 3.2 - Chunking

- Use `RecursiveCharacterTextSplitter`, `create_documents`, and `split_documents`.
- Preserve each PubMed document ID in `metadatas`.
- Print chunk samples and answer the chunking reflection.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1_000
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    is_separator_regex=False,
)

metadatas = [{"id": str(idx)} for idx in documents.index]

texts = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)
text_chunks = text_splitter.split_documents(texts)

assert text_chunks
assert all("id" in chunk.metadata for chunk in text_chunks)

print(f"Source documents: {len(documents):,}")
print(f"LangChain documents after create_documents: {len(texts):,}")
print(f"Final text chunks after split_documents: {len(text_chunks):,}")

for i, chunk in enumerate(text_chunks[:3]):
    print(f"\nChunk {i + 1} | metadata={chunk.metadata} | characters={len(chunk.page_content)}")
    print(chunk.page_content)


Source documents: 890
LangChain documents after create_documents: 1,928
Final text chunks after split_documents: 1,928

Chunk 1 | metadata={'id': '21645374'} | characters=997
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo

**Reflection:** Chunk size controls the trade-off between precise retrieval and enough context to answer. Overlap reduces information loss at boundaries, but excessive overlap creates duplicate evidence, increases storage, and can crowd the prompt.


## Task 3.3 - Define a vector store

Create a Chroma vector store using cosine distance, the Task 3.1 embedding model, and the Task 3.2 chunks. Run the required programmed-cell-death query with `k=3`.


In [6]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=text_chunks,
    embedding=embedding_model,
    collection_name="pubmedqa_assignment4_full",
    collection_metadata={"hnsw:space": "cosine"},
)

results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)

for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")


* [SIM=0.538406] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTrac

## Task 4.1 - Define the full RAG pipeline

Implement **Option B** with an LCEL retriever, `ChatPromptTemplate`, `RunnableParallel`, `StrOutputParser`, and `assign`. Retrieve one document per question and return both the context and answer.


In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

retriever = vector_store.as_retriever(search_kwargs={"k": 1})

rag_template = """
Use the retrieved PubMed context to answer the medical question.
Return exactly one word: Yes or No.

Retrieved context:
{context}

Question: {question}
Answer:
"""

prompt = ChatPromptTemplate.from_template(rag_template)

runnable_parallel_object = RunnableParallel(
    context=retriever,
    question=RunnablePassthrough(),
)

chain = prompt | model | StrOutputParser()
rag_chain = runnable_parallel_object.assign(answer=chain)

your_query = questions.iloc[0].question
sanity_result = rag_chain.invoke(your_query)

print("Question:", your_query)
print("Gold label:", questions.iloc[0].gold_label)
print("Retrieved document ID:", sanity_result["context"][0].metadata.get("id"))
print("Answer:", sanity_result["answer"])
print("Retrieved text:\n", sanity_result["context"][0].page_content)


[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold label: yes
Retrieved document ID: 21645374
Answer: Yes


Assistant: Yes

The retrieved
Retrieved text:
 Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression 

## Task 5.1 - High-level evaluation

Evaluate the full RAG pipeline on every retained question. Report valid-answer count, accuracy, and F1, then run the same language model without context and compare the two setups.


In [8]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm


def parse_yes_no(text):
    match = re.match(r"^\s*(yes|no)\b", str(text), flags=re.IGNORECASE)
    return match.group(1).lower() if match else None


def run_full_dataset_in_batches(runnable, inputs, batch_size=64, description="Running"):
    outputs = []
    for start in tqdm(range(0, len(inputs), batch_size), desc=description):
        batch = inputs[start : start + batch_size]
        outputs.extend(runnable.batch(batch, config={"max_concurrency": 4}))
    return outputs


def classification_metrics(gold, predictions):
    gold = pd.Series(gold).reset_index(drop=True).astype(str).str.lower()
    predictions = pd.Series(predictions).reset_index(drop=True)
    valid = predictions.isin(["yes", "no"])
    valid_count = int(valid.sum())
    if valid_count == 0:
        return {
            "valid_answers": 0,
            "total_questions": len(gold),
            "valid_fraction": 0.0,
            "accuracy": np.nan,
            "f1_yes": np.nan,
        }
    return {
        "valid_answers": valid_count,
        "total_questions": len(gold),
        "valid_fraction": valid_count / len(gold),
        "accuracy": accuracy_score(gold[valid], predictions[valid]),
        "f1_yes": f1_score(
            gold[valid], predictions[valid], pos_label="yes", zero_division=0
        ),
    }


question_texts = questions.question.astype(str).tolist()
gold_labels = questions.gold_label.astype(str).str.lower().tolist()

start_time = time.time()
rag_outputs = run_full_dataset_in_batches(
    rag_chain,
    question_texts,
    description="Full RAG evaluation",
)
rag_seconds = time.time() - start_time

baseline_template = """
Answer the medical question using your own knowledge and no retrieved context.
Return exactly one word: Yes or No.

Question: {question}
Answer:
"""
baseline_prompt = ChatPromptTemplate.from_template(baseline_template)
baseline_chain = baseline_prompt | model | StrOutputParser()

baseline_inputs = [{"question": question} for question in question_texts]
start_time = time.time()
baseline_outputs = run_full_dataset_in_batches(
    baseline_chain,
    baseline_inputs,
    description="Full no-context evaluation",
)
baseline_seconds = time.time() - start_time

rag_raw_answers = [output["answer"] for output in rag_outputs]
rag_predictions = [parse_yes_no(answer) for answer in rag_raw_answers]
baseline_predictions = [parse_yes_no(answer) for answer in baseline_outputs]

rag_metrics = classification_metrics(gold_labels, rag_predictions)
baseline_metrics = classification_metrics(gold_labels, baseline_predictions)

metrics_table = pd.DataFrame(
    [
        {"setup": "RAG", **rag_metrics, "runtime_seconds": rag_seconds},
        {
            "setup": "Same LM without context",
            **baseline_metrics,
            "runtime_seconds": baseline_seconds,
        },
    ]
)
display(metrics_table)

accuracy_change = rag_metrics["accuracy"] - baseline_metrics["accuracy"]
f1_change = rag_metrics["f1_yes"] - baseline_metrics["f1_yes"]
print(f"Accuracy change from retrieval: {accuracy_change:+.4f}")
print(f"F1 change from retrieval: {f1_change:+.4f}")
if pd.notna(accuracy_change) and pd.notna(f1_change):
    if accuracy_change > 0 and f1_change > 0:
        print("Conclusion: retrieval helped on both reported classification metrics.")
    elif accuracy_change < 0 and f1_change < 0:
        print("Conclusion: retrieval did not help on either reported classification metric.")
    else:
        print("Conclusion: retrieval had mixed effects across accuracy and F1.")


Full RAG evaluation:   0%|          | 0/14 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/tra

Full no-context evaluation:   0%|          | 0/14 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/tra

,setup,valid_answers,total_questions,valid_fraction,accuracy,f1_yes,runtime_seconds
0,RAG,890,890,1.0,0.624719,0.766434,377.178225
1,Same LM without context,890,890,1.0,0.568539,0.677311,70.391948


Accuracy change from retrieval: +0.0562
F1 change from retrieval: +0.0891
Conclusion: retrieval helped on both reported classification metrics.


## Task 5.2 - Detailed inspection

Compare the retrieved document ID with `questions.gold_document_id` for every question. Then inspect retrieved documents and corresponding answers.


In [9]:
retrieved_document_ids = [
    str(output["context"][0].metadata.get("id")) if output["context"] else None
    for output in rag_outputs
]

evaluation = questions.reset_index(drop=True).copy()
evaluation["rag_raw_answer"] = rag_raw_answers
evaluation["rag_prediction"] = rag_predictions
evaluation["baseline_raw_answer"] = baseline_outputs
evaluation["baseline_prediction"] = baseline_predictions
evaluation["retrieved_document_id"] = retrieved_document_ids
evaluation["gold_document_id"] = evaluation.gold_document_id.astype(str)
evaluation["gold_document_fetched"] = (
    evaluation.retrieved_document_id == evaluation.gold_document_id
)
evaluation["rag_correct"] = evaluation.rag_prediction == evaluation.gold_label
evaluation["baseline_correct"] = (
    evaluation.baseline_prediction == evaluation.gold_label
)

gold_document_fetch_rate = evaluation.gold_document_fetched.mean()
print(
    f"Gold documents fetched at rank 1: "
    f"{evaluation.gold_document_fetched.sum():,}/{len(evaluation):,} "
    f"({gold_document_fetch_rate:.2%})"
)

display(
    evaluation[
        [
            "question",
            "gold_label",
            "rag_prediction",
            "baseline_prediction",
            "gold_document_id",
            "retrieved_document_id",
            "gold_document_fetched",
        ]
    ].head(10)
)


def first_position(mask):
    positions = np.flatnonzero(np.asarray(mask, dtype=bool))
    return int(positions[0]) if len(positions) else None


inspection_positions = []
for candidate in [
    first_position(evaluation.gold_document_fetched),
    first_position(~evaluation.gold_document_fetched),
    first_position(evaluation.rag_correct),
    first_position(~evaluation.rag_correct),
]:
    if candidate is not None and candidate not in inspection_positions:
        inspection_positions.append(candidate)

for position in inspection_positions[:4]:
    row = evaluation.iloc[position]
    retrieved_doc = rag_outputs[position]["context"][0]
    print("\n" + "=" * 100)
    print("Question:", row.question)
    print("Gold label:", row.gold_label)
    print("RAG answer:", row.rag_raw_answer)
    print("No-context answer:", row.baseline_raw_answer)
    print("Gold document ID:", row.gold_document_id)
    print("Retrieved document ID:", row.retrieved_document_id)
    print("Gold document fetched:", row.gold_document_fetched)
    print("Retrieved document:\n", retrieved_doc.page_content)

OUTPUT_DIR = Path("/content/a4_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
evaluation.to_csv(OUTPUT_DIR / "assignment4_full_evaluation.csv", index=False)

summary = {
    "model_id": MODEL_ID,
    "embedding_model_id": EMBEDDING_MODEL_ID,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "retriever_k": 1,
    "number_of_questions": len(evaluation),
    "rag_metrics": rag_metrics,
    "baseline_metrics": baseline_metrics,
    "gold_document_fetch_rate_at_1": gold_document_fetch_rate,
}
with open(OUTPUT_DIR / "assignment4_metrics.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print("\nSaved:", OUTPUT_DIR / "assignment4_full_evaluation.csv")
print("Saved:", OUTPUT_DIR / "assignment4_metrics.json")


Gold documents fetched at rank 1: 866/890 (97.30%)


,question,gold_label,rag_prediction,baseline_prediction,gold_document_id,retrieved_document_id,gold_document_fetched
0,Do mitochondria play a role in remodelling lac...,yes,yes,yes,21645374,21645374,True
1,Landolt C and snellen e acuity: differences in...,no,yes,yes,16418930,16418930,True
2,"Syncope during bathing in infants, a pediatric...",yes,yes,yes,9488747,9488747,True
3,Are the long-term results of the transanal pul...,no,yes,no,17208539,17208539,True
4,Can tailored interventions increase mammograph...,yes,yes,yes,10808977,10808977,True
5,Double balloon enteroscopy: is it efficacious ...,yes,yes,yes,23831910,25251991,False
6,Is adjustment for reporting heterogeneity nece...,no,yes,yes,26852225,26852225,True
7,Do mutations causing low HDL-C promote increas...,no,yes,no,17113061,17113061,True
8,A short stay or 23-hour ward in a general and ...,yes,yes,yes,10966337,10966337,True
9,Did Chile's traffic law reform push police enf...,yes,yes,no,25432938,25432938,True



Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold label: yes
RAG answer: Yes


Assistant: Yes

The retrieved
No-context answer: Yes. Mitochondrial activity is known
Gold document ID: 21645374
Retrieved document ID: 21645374
Gold document fetched: True
Retrieved document:
 Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariens